In [26]:
!curl -sSL https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!./.venv/bin/python get-pip.py
!rm get-pip.py

zsh:1: no such file or directory: ./.venv/bin/python


In [27]:
%pip install docling llama-index-readers-docling
%pip install -qU pip docling transformers

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [28]:
%pip install llama_index.node_parser.docling

Note: you may need to restart the kernel to use updated packages.


1) FILE PARSE

In [29]:
from docling.chunking import HybridChunker
from llama_index.readers.docling import DoclingReader
from llama_index.node_parser.docling import DoclingNodeParser
from transformers import AutoTokenizer


file_path = "./sample3BM.pdf"

docling_reader = DoclingReader(
    export_type = DoclingReader.ExportType.JSON
)

file = docling_reader.load_data(file_path=[file_path])




[INFO] 2026-05-20 08:36:40,762 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-20 08:36:40,765 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-20 08:36:40,779 [RapidOCR] download_file.py:60: File exists and is valid: /Users/raziqs/Desktop/MyKepatuhan/MyKepatuhan/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-20 08:36:40,779 [RapidOCR] main.py:50: Using /Users/raziqs/Desktop/MyKepatuhan/MyKepatuhan/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-20 08:36:40,957 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-20 08:36:40,958 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-20 08:36:40,961 [RapidOCR] download_file.py:60: File exists and is valid: /Users/raziqs/Desktop/MyKepatuhan/MyKepatuhan/.venv/lib/python3.13/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-20 08:36:40,961 [RapidOCR] main.py:50: Using /Us

2) HYBRID CHUNKING

In [30]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from dotenv import load_dotenv
load_dotenv()

EMBED_MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAXTOKEN = 512

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL),
    max_tokens=MAXTOKEN
)

chunker = HybridChunker(
    tokenizer=tokenizer
)

node_parser = DoclingNodeParser(
    chunker=chunker
)

nodes = node_parser.get_nodes_from_documents(file)

print(f"Created {len(nodes)} nodes from the document.")

Created 2 nodes from the document.


In [31]:
for i, node in enumerate(nodes):
    # Retrieve the element type from Docling's metadata
    # (Keys vary slightly by version, commonly 'dl_meta' or direct keys)
    meta = node.metadata
    
    # 1. Check for Tables
    if "table" in str(meta).lower() or meta.get("label") == "Table":
        print(f"Node {i} contains a TABLE.")
        print(f"Content:\n{node.text}\n\n") # Usually outputs Markdown/CSV text representation

    # 2. Check for Pictures / Figures
    elif "picture" in str(meta).lower() or meta.get("label") == "Picture":
        print(f"Node {i} contains a PICTURE/IMAGE.")
        print(f"Caption/Description: {node.text}")

In [32]:
%pip install llama-index-llms-ollama



Note: you may need to restart the kernel to use updated packages.


ADD additional metadata

In [33]:
from llama_index.llms.ollama import Ollama
import json

llm = Ollama(
    model="gemma4:e4b",
    temperature=0.1,
    request_timeout=120
)

prompt_template = """
You are an expert Malaysian corporate lawyer. Read the following text chunk and extract the metadata.
You MUST respond ONLY with a valid JSON object matching this exact format. Do not include markdown formatting or explanations.

{{
  "jurisdiction": "Choose ONE: federal, state, local, or unknown or anything that relates to the jurisdiction of the document",
  "authority": "Choose ONE: SSM, KKM, DBKL, MPKj, LHDN, or unknown or anything that relates to the authority that issued the document",
  "topic": "Choose ONE: tax, licensing, zoning, employment, registration, or unknown or anything that relates to the topic of the document",
  "document_type": "Choose ONE: act, guideline, form, fee_schedule, or unknown or anything that relates to the type of the document"
}}

TEXT TO ANALYZE:
{chunk_text}
"""

print("Generating metadata for each node...")

for i, node in enumerate(nodes):
    print(f"Processing Node {i+1}...")
    prompt = prompt_template.format(chunk_text=node.text)

    try:
        response = llm.complete(prompt)
        print(f"reponse to node {i+1}")

        extracted_metadata = json.loads(response.text)
        print(f"Extracted Metadata for Node {i+1}")

        node.metadata.update(extracted_metadata)
        print(f"Successfully updated node metadata {i+1}.\n")  


    except Exception as e:
        print(f"Error processing Node {i+1}: {e}\n")

print("Metadata generation complete.")


Generating metadata for each node...
Processing Node 1...
reponse to node 1
Extracted Metadata for Node 1
Successfully updated node metadata 1.

Processing Node 2...
reponse to node 2
Extracted Metadata for Node 2
Successfully updated node metadata 2.

Metadata generation complete.


In [34]:
%pip install llama_index-embeddings-ollama llama-index-vector-stores-pinecone pinecone-client

Note: you may need to restart the kernel to use updated packages.


Embedding and sent to vector store

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import json

from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import VectorStoreIndex, Settings, StorageContext
from pinecone import Pinecone

for node in nodes:
    # We iterate over a copy of the keys so we can modify the dict safely
    for key, value in list(node.metadata.items()):
        # If the value is a complex type (list or dict), convert it to a string
        if isinstance(value, (dict, list)):
            node.metadata[key] = json.dumps(value)
        # Pinecone also rejects 'None' values, so we replace them with empty strings
        elif value is None:
            node.metadata[key] = ""

print("Metadata sanitized successfully!")

PINECONE_API_KEY = os.getenv("PINECON_KEY")

embed_model = OllamaEmbedding(
    model_name="nomic-embed-text-v2-moe",
    embed_batch_size=100,
)

Settings.embed_model = embed_model

print("Connecting to Pinecone...")

pc = Pinecone(api_key=PINECONE_API_KEY)
pinecone_index = pc.Index("mykepatuhan")

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


index = VectorStoreIndex(
    nodes=nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True
)

print("SUCCESS! Check your Pinecone Dashboard, the vectors should be there.")


Metadata sanitized successfully!
Connecting to Pinecone...


Upserted vectors: 100%|██████████| 2/2 [00:01<00:00,  1.05it/s]

SUCCESS! Check your Pinecone Dashboard, the vectors should be there.
